# bias-correction-divide — ex2: 10-step bias-correction trajectory + convergence plot

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `bias-correction-divide`. Running the final beacon cell reports progress against the `Optimizer: Adam bias-correction divide` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: Adam bias-correction divide` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`bias-correction-divide`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "bias-correction-divide"
DD_SUBTOPIC = "Optimizer: Adam bias-correction divide"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Bias-correction trajectory across steps — quick refresher

For constant gradient `g` the raw EMA converges to `g` from zero:
```
m_t = (1 - beta**t) * g          # raw EMA, biased toward 0
m_hat_t = m_t / (1 - beta**t)    # = g exactly, for all t >= 1
```
The correction factor `1 / (1 - beta**t)` is HUGE early (e.g. step 1 with beta=0.9 → 10.0×) and fades to 1.0 as `t` grows. Plotted on a per-step axis, `m_hat` is a flat horizontal line at `g` from step 1 onwards while raw `m` only ramps in slowly.

```python
m = 0.0; beta = 0.9
for tt in range(1, 11):
    m = beta*m + (1-beta)*g                    # raw EMA
    m_hat = m / (1 - beta**tt)                 # bias-corrected
```

### Exercise 2 — 10-step bias-correction trajectory + convergence plot

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the EMA recurrence + bias-correction divide across 10 steps of a constant gradient to produce TWO trajectories — raw `m` (biased) and `m_hat` (unbiased, flat at `g`).
> Keywords: bias-correction, trajectory, convergence, visualization
> ```

**KCs targeted:** `bias-correction-divide-by-one-minus-beta-power-t`, `ema-converges-to-constant-g`

Implement `ex2_bias_correction_trajectory(g, beta, n_steps)`. Run the Adam EMA + bias-correction for `n_steps` steps on a CONSTANT scalar gradient `g`.

1. Start `m = 0.0` (Python float, scalar EMA buffer).
2. For `step in range(1, n_steps + 1)`:
   - `m = beta * m + (1 - beta) * g` — raw EMA update.
   - `m_hat = m / (1 - beta ** step)` — bias-corrected.
   - Append `m` to `raw_history`, `m_hat` to `corrected_history`.
3. Return `(raw_history, corrected_history)` as two lists of floats, each of length `n_steps`.

The test verifies:
- Raw `m` ramps from `(1-beta)*g` toward `g`, never reaching it.
- `m_hat` is EXACTLY `g` at every step (within float tolerance) for constant `g`.
- A matplotlib plot of both trajectories is rendered (headless).

In [ ]:
def ex2_bias_correction_trajectory(g, beta, n_steps):
    m = 0.0
    raw_history = []
    corrected_history = []
    for step in range(1, n_steps + 1):
        m = beta * m + (1 - beta) * g
        m_hat = m / (1 - beta ** step)
        raw_history.append(float(m))
        corrected_history.append(float(m_hat))
    return raw_history, corrected_history


<details><summary>Solution</summary>

```python
def ex2_bias_correction_trajectory(g, beta, n_steps):
    m = 0.0
    raw_history = []
    corrected_history = []
    for step in range(1, n_steps + 1):
        m = beta * m + (1 - beta) * g
        m_hat = m / (1 - beta ** step)
        raw_history.append(float(m))
        corrected_history.append(float(m_hat))
    return raw_history, corrected_history
```

**The plot is the insight.** Raw `m` ramps in slowly (still 65% of `g` at step 10 with beta=0.9); `m_hat` is flat at `g` from step 1. That flat line is what makes Adam usable from the first step — without correction, the warmup bias would silently slow training for ~100 steps with beta1=0.9.

**Why scalar instead of tensor.** Same recurrence, no advantage to tensors here — the point is the trajectory math. A real Adam implementation does this per-element on the full Parameter tensor, but the per-step factor `(1 - beta**t)` is a scalar.

**Generalization to non-constant g.** With a time-varying `g_t`, `m_hat_t` won't equal `g_t` exactly — it's a low-pass-filtered estimate. The bias correction still removes the zero-init bias, but the EMA filter still attenuates high-frequency content. That's the desired behavior.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()